# ALEsys PDF Ingestion (Colab Mode)

Este notebook ejecuta el pipeline de ingesta PDF de ALEsys en modo `files_only` (sin GraphRAG), compatible con Google Colab.

In [ ]:
# @title Setup Environment
# @markdown Instalar dependencias y configurar paths

!pip install magic-pdf pdfplumber pymupdf -q
import os
from pathlib import Path

OUTPUT_DIR = Path("/content/alesys-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# @title Upload PDF
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}")

In [ ]:
# @title Configure MinerU
import json

config = {
    "mode": "files_only",
    "topic": "colab-ingest",
    "ocr_languages": ["en", "es"],
    "extract_formulas": True,
    "extract_tables": True
}
print(json.dumps(config, indent=2))

In [ ]:
# @title Run Ingestion via API
import requests
import time

API_URL = os.environ.get("ALESYS_API_URL", "http://localhost:8080/api/v1")

payload = {
    "pdf_path": f"/content/{pdf_path}",
    "topic": "colab-ingest",
    "mode": "files_only",
    "force_fallback": False,
    "ocr_languages": ["en", "es"],
    "extract_formulas": True,
    "extract_tables": True
}

try:
    response = requests.post(f"{API_URL}/ingestion/pdf", json=payload, timeout=120)
    result = response.json()
    print(f"Job ID: {result.get('job_id', 'N/A')}")
    print(f"Success: {result.get('success', False)}")
    if result.get('output_dir'):
        print(f"Output: {result['output_dir']}")
except Exception as e:
    print(f"API not available (expected for local): {e}")
    print("Usando MinerU directamente...")

In [ ]:
# @title Run MinerU Directly (Fallback)
import subprocess
import shutil

book_name = Path(pdf_path).stem
output_path = OUTPUT_DIR / book_name

cmd = [
    "magic-pdf",
    f"/content/{pdf_path}",
    f"--output-dir={output_path}",
    "--method=ocr"
]

print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)

In [ ]:
# @title Organize Output (port de reordenar_db_p.py)
def organize_output(output_path: Path):
    """Port simplificado de reordenar_db_p.py"""
    import re
    
    # Find MD files
    md_files = list(output_path.rglob("*.md"))
    print(f"Found {len(md_files)} markdown files")
    
    for md_file in md_files:
        content = md_file.read_text()
        # Extract image references
        img_refs = re.findall(r'!\[.*?\]\(([^)]+)\)', content)
        
        # Create organized dir
        organized = output_path / "organized" / md_file.stem
        organized.mkdir(parents=True, exist_ok=True)
        
        # Move referenced images
        for ref_path in img_refs:
            src = output_path / ref_path
            if src.exists():
                dst = organized / "images" / src.name
                dst.parent.mkdir(exist_ok=True)
                shutil.copy2(src, dst)
                print(f"Copied: {src.name} → {dst}")
        
        # Copy MD
        shutil.copy2(md_file, organized / f"{md_file.stem}.md")
    
    return output_path / "organized"

organized_path = organize_output(output_path)
print(f"Organized files in: {organized_path}")

In [ ]:
# @title Download Results
!zip -r /content/result.zip {organized_path} -q
files.download("/content/result.zip")